# Distributed Storage Benchmarking — Measure Copy Throughput

This notebook benchmarks distributed shared storage write throughput across multiple FABRIC sites using various copy methods (Python, `rsync`, `dd`, `pv`).

It uses `storage=True` to automatically provision shared storage on each node, then runs benchmarks comparing shared storage vs local disk performance.

## What it measures

* **Python buffered copy** (portable baseline)
* **rsync** (`--inplace`) sequential write
* **dd** sequential write
* Optional **pv|dd** progress variant (if available)

Results are recorded as CSV and plotted for comparison across sites/methods.

> **New to distributed storage on FABRIC?** See the [Distributed Storage tutorial](../../fablib_api/cephfs_storage/cephfs_storage.ipynb) for getting started with shared storage.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

## Create Slice with Distributed Storage

Using `storage=True`, FABlib automatically adds FABNetv4, generates storage credentials, installs required packages, and mounts shared storage on every node during post-boot.

In [ ]:
slice_name = "ceph-benchmarking"

try:
    slice = fablib.get_slice(slice_name)
    print(f"Slice '{slice_name}' already exists, reusing it.")
    slice.list_nodes()
except Exception:
    slice = fablib.new_slice(name=slice_name, storage=True)

    # Place nodes at 3 random sites for distance-based comparison
    sites = fablib.get_random_sites(3)

    for site in sites:
        slice.add_node(
            name=f"{site.lower()}-client",
            cores=4,
            ram=16,
            disk=500,
            site=site,
        )

    slice.submit()

In [ ]:
slice = fablib.get_slice(name=slice_name)

for node in slice.get_nodes():
    print(f"--- {node.get_name()} (storage={node.has_storage()}, cluster={node.get_storage_cluster()}) ---")
    node.execute("df -h | grep ceph")
    print()

## Run the tests

Execute the copy benchmarks against the selected mount. The runner measures:

* **Python buffered copy** (portable baseline)
* **rsync** (`--inplace`) and **dd** (sequential write)
* Optional **pv|dd** or **tqdm** progress variant (if available)

Each test reports MB/s/GiB/s and elapsed time, then appends a row to
`<mount>/benchmarks/results/<hostname>_<timestamp>.csv`.

**Note:** Throughput and latency can vary with the client's **proximity to the storage cluster** (RTT, cross-region hops, peering), as well as host NIC/CPU, MTU, and time-of-day congestion. Closer sites typically show higher throughput and lower variance.


In [ ]:
from pathlib import Path
from datetime import datetime

LOCAL_RESULTS = Path("results_csv")
SIZE_GB = 2
METHODS = "python,rsync,dd,pv"            # change as you like

LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

# Install pv on each node for the pv benchmark method
for node in slice.get_nodes():
    node.execute("sudo dnf install -y pv 2>/dev/null || sudo apt-get install -y -qq pv 2>/dev/null || true", quiet=True)

for node in slice.get_nodes():
    node_name = node.get_name()
    label = node_name
    CSV_GLOB = f"benchmarks/results/{node_name}*.csv"
    print(f"=== {node_name} ===")

    # 1) Push benchmark tool
    node.upload_directory("node_tools", ".")

    # 2) Discover mount points from the live node
    stdout, _ = node.execute("mount | grep ceph | awk '{print $3}'", quiet=True)
    mount_points = [line.strip() for line in stdout.splitlines() if line.strip()]

    if not mount_points:
        print(f"[WARN] No CephFS mounts found on {node_name}, skipping.")
        continue

    for mount_point in mount_points:
        # Run the benchmark remotely
        cmd = (
            f"python3 node_tools/write_bench.py "
            f"--mount '{mount_point}' --size-gb {SIZE_GB} "
            f"--methods '{METHODS}' --label '{label}' "
            f"--compare-local"
        )
        stdout, stderr = node.execute(cmd)
        print(stdout or "")

        # 3) List CSVs on remote
        list_cmd = f"ls -1 {mount_point}/{CSV_GLOB} 2>/dev/null || true"
        stdout, stderr = node.execute(list_cmd, quiet=True)
        csv_paths = [line.strip() for line in (stdout or "").splitlines() if line.strip().endswith(".csv")]
        if not csv_paths:
            print(f"[INFO] No CSVs found at {mount_point}/{CSV_GLOB} on {node_name}")
            continue

        # 4) Download each CSV to results_csv
        LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)
        print(f"Downloading results")
        for remote_csv in csv_paths:
            basename = Path(remote_csv).name
            local_csv = LOCAL_RESULTS / basename
            node.download_file(str(local_csv), remote_csv)

## Results Table & Plot

Aggregates all CSV results (across nodes/methods), computes MB/s/GiB/s, and shows a sortable table (timestamp → host → method).
Includes a grouped bar chart by host/label and method, plus a variability summary (e.g., boxplot).

**Interpretation tip:** Compare hosts by region/label to see **distance effects**—nodes farther from the storage cluster (higher RTT, more WAN hops) usually exhibit lower write rates and longer tails.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- soft styling defaults (matplotlib only) ---
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f9",
    "axes.edgecolor": "#dddddd",
    "axes.grid": True,
    "grid.color": "#e6e6ee",
    "grid.linestyle": "-",
    "grid.linewidth": 0.6,
    "axes.titleweight": "semibold",
    "axes.titlepad": 10,
    "axes.labelpad": 6,
    "xtick.color": "#444444",
    "ytick.color": "#444444",
    "legend.frameon": False,
})

# a calm, repeating pastel palette (color-blind friendly leaning)
SOOTHING = [
    "#8ECAE6",  # soft blue
    "#BDE0FE",  # very light blue
    "#A7D9C9",  # mint
    "#CFE1B9",  # sage
    "#FFD6A5",  # peach
    "#FFECB3",  # pale amber
    "#D7E3FC",  # powder blue
    "#EAD7F6",  # lavender
    "#F6D6D6",  # blush
    "#C9E4DE",  # seafoam
]

def color_cycle_for(methods):
    # deterministic map: method -> color (repeat if needed)
    methods = list(methods)
    cmap = {m: SOOTHING[i % len(SOOTHING)] for i, m in enumerate(sorted(methods))}
    return cmap

# Fixed colors for targets in side-by-side charts
COLOR_CEPH = "#8ECAE6"  # soft blue
COLOR_LOCAL = "#CFE1B9" # sage

RESULTS_DIR = Path("./results_csv")  # where you scp'd the CSVs
csv_files = sorted(RESULTS_DIR.glob("*.csv"))
assert csv_files, "No CSV files found in results_csv/"

dfs = [pd.read_csv(p) for p in csv_files]
df = pd.concat(dfs, ignore_index=True)

# ---- Normalize / clean fields ----
df['hostname']   = df['hostname'].astype(str)
df['label']      = df.get('label', "").fillna("").astype(str)
df['method']     = df['method'].astype(str)
df['MB_per_s']   = pd.to_numeric(df['MB_per_s'], errors='coerce')
df['GiB_per_s']  = pd.to_numeric(df.get('GiB_per_s', df['MB_per_s'] / 1024.0), errors='coerce')
df['elapsed_s']  = pd.to_numeric(df['elapsed_s'], errors='coerce')
df['size_bytes'] = pd.to_numeric(df['size_bytes'], errors='coerce')
df['size_gib']   = df['size_bytes'] / (1024**3)

# Backward-compat: if 'target' missing, assume 'cephfs'
if 'target' not in df.columns:
    df['target'] = 'cephfs'
else:
    df['target'] = df['target'].fillna('cephfs').astype(str)

# --------------- Helpers ----------------
def grouped_bar_by_host(target_df: pd.DataFrame, title_suffix: str):
    pivot = target_df.pivot_table(index=['hostname','label'], columns='method',
                                  values='MB_per_s', aggfunc='mean')
    if pivot.empty:
        print(f"[plot] No data for {title_suffix}; skipping plot.")
        return
    pivot = pivot.sort_index()
    methods = list(pivot.columns)
    colors = color_cycle_for(methods)

    hosts   = [' / '.join([h, l]) if l else h for h,l in pivot.index]
    x = np.arange(len(hosts))
    width = 0.8 / max(1, len(methods))

    plt.figure(figsize=(min(14, 0.4*len(hosts)+6), 5))
    for i, m in enumerate(methods):
        vals = pivot[m].fillna(0).values
        offs = x + i*width - (len(methods)-1)*width/2
        plt.bar(offs, vals, width, label=m, color=colors[m])

    plt.xticks(x, hosts, rotation=30, ha='right')
    plt.ylabel("Throughput (MB/s)")
    plt.title(f"Copy Throughput by Host and Method (mean) — {title_suffix}")
    plt.legend(ncols=min(4, len(methods)))
    plt.tight_layout()
    plt.show()

def boxplot_by_method(target_df: pd.DataFrame, title_suffix: str):
    methods = sorted(target_df['method'].unique().tolist())
    series = [target_df.loc[target_df['method']==m, 'MB_per_s'].dropna().values for m in methods]
    if all(len(s) == 0 for s in series):
        print(f"[plot] No data for {title_suffix}; skipping boxplot.")
        return

    colors = color_cycle_for(methods)
    plt.figure(figsize=(min(14, 0.6*len(methods)+4), 5))
    bp = plt.boxplot(series, tick_labels=methods, showmeans=True, patch_artist=True)

    # apply soothing colors to boxes/medians/means
    for patch, m in zip(bp['boxes'], methods):
        patch.set_facecolor(colors[m])
        patch.set_alpha(0.85)
        patch.set_edgecolor("#bbbbbb")
    for median in bp['medians']:
        median.set_color("#6b6b6b")
        median.set_linewidth(1.6)
    for mean in bp['means']:
        mean.set_marker('o')
        mean.set_markerfacecolor("#444444")
        mean.set_markeredgecolor = "#444444"

    plt.ylabel("Throughput (MB/s)")
    plt.title(f"Distribution of Throughput by Method — {title_suffix}")
    plt.tight_layout()
    plt.show()

def paired_speedup(all_df: pd.DataFrame) -> pd.DataFrame:
    have_both = set(all_df['target'].unique()) >= {'cephfs','local'}
    if not have_both:
        print("[paired] Need both 'cephfs' and 'local' targets to compute speedup; skipping.")
        return pd.DataFrame()

    key_cols = ['hostname','label','method','size_bytes']
    ceph = (all_df[all_df['target']=='cephfs']
            .groupby(key_cols, as_index=False)['MB_per_s'].mean()
            .rename(columns={'MB_per_s':'cephfs_MBps'}))
    loc  = (all_df[all_df['target']=='local']
            .groupby(key_cols, as_index=False)['MB_per_s'].mean()
            .rename(columns={'MB_per_s':'local_MBps'}))

    merged = pd.merge(ceph, loc, on=key_cols, how='inner')
    if merged.empty:
        print("[paired] No overlapping (hostname,label,method,size_bytes) between cephfs/local.")
        return pd.DataFrame()

    merged['size_gib'] = merged['size_bytes'] / (1024**3)
    # Avoid divide-by-zero
    merged['speedup_local_over_cephfs'] = merged.apply(
        lambda r: np.nan if r['cephfs_MBps'] == 0 else r['local_MBps'] / r['cephfs_MBps'], axis=1
    )
    return merged

def plot_speedup_sorted(speed_df: pd.DataFrame):
    """
    Bar chart of speedup (Local/CephFS) sorted descending, with 1.0 baseline,
    value labels, and concise x tick labels (host[/label] • method).
    """
    if speed_df.empty:
        print("[plot] No paired data to plot speedup.")
        return

    work = speed_df.copy()
    work = work.dropna(subset=['speedup_local_over_cephfs'])
    if work.empty:
        print("[plot] No finite speedup values to plot.")
        return

    work['xlab'] = work.apply(
        lambda r: (f"{r['hostname']}" + (f"/{r['label']}" if r['label'] else "") + f" • {r['method']}"),
        axis=1
    )
    work = work.sort_values('speedup_local_over_cephfs', ascending=False)

    x = np.arange(len(work))
    vals = work['speedup_local_over_cephfs'].values

    plt.figure(figsize=(min(16, 0.6*len(work)+5), 5))
    bars = plt.bar(x, vals, width=0.8, color="#CFE1B9", label="Local / CephFS (mean)")

    # baseline at 1.0
    plt.axhline(1.0, linestyle='--', color="#9aa0a6", linewidth=1.2, label="Parity (1.0)")

    # data labels
    for i, b in enumerate(bars):
        h = b.get_height()
        plt.text(b.get_x() + b.get_width()/2, h + 0.02, f"{h:.2f}×", ha='center', va='bottom', fontsize=8, color="#444444")

    plt.xticks(x, work['xlab'].tolist(), rotation=40, ha='right')
    plt.ylabel("Speedup factor (Local ÷ CephFS)")
    plt.title("Write Speedup — Local vs CephFS (sorted)")
    plt.ylim(0, max(1.1, np.nanmax(vals)*1.15))
    plt.legend()
    plt.tight_layout()
    plt.show()

# -------- NEW: Side-by-side Local vs CephFS (per method) --------
def side_by_side_local_cephfs(all_df: pd.DataFrame, method: str | None = None):
    """
    For each host(/label), show Local vs CephFS bandwidth (MB/s) as adjacent vertical bars
    on the same axes. Optionally filter to a single method.
    """
    required = {'local', 'cephfs'}
    if not (set(all_df['target'].unique()) >= required):
        print("[compare] Need both 'local' and 'cephfs' to compare side-by-side.")
        return

    data = all_df.copy()
    if method is not None:
        data = data[data['method'] == method]
        if data.empty:
            print(f"[compare] No rows for method={method!r}.")
            return

    # Aggregate mean MB/s per host/label/target (keeping method for labeling)
    key = ['hostname', 'label', 'method', 'target']
    grp = (data.groupby(key, as_index=False)['MB_per_s'].mean()
                .rename(columns={'MB_per_s': 'MBps_mean'}))

    if grp.empty:
        print("[compare] No data after grouping.")
        return

    # If method not specified, pick the densest method for a single clean chart
    if method is None and grp['method'].nunique() > 1:
        densest = grp.groupby('method').size().sort_values(ascending=False).index[0]
        grp = grp[grp['method'] == densest].copy()
        subtitle_method = f" (method = {densest})"
    else:
        subtitle_method = f" (method = {method})" if method else ""

    # pivot to have columns: 'cephfs' and 'local'
    piv = grp.pivot_table(index=['hostname','label'], columns='target', values='MBps_mean', aggfunc='mean')
    # Keep only rows that have both targets
    piv = piv.dropna(subset=['cephfs', 'local'], how='any')
    if piv.empty:
        print("[compare] No host/label entries with both local and cephfs.")
        return
    piv = piv.sort_index()

    labels = [' / '.join([h, l]) if l else h for h, l in piv.index]
    x = np.arange(len(labels))
    width = 0.38

    plt.figure(figsize=(min(16, 0.5*len(labels)+6), 5))
    b1 = plt.bar(x - width/2, piv['cephfs'].values, width, label='CephFS (mean MB/s)', color=COLOR_CEPH)
    b2 = plt.bar(x + width/2, piv['local'].values,  width, label='Local (mean MB/s)',  color=COLOR_LOCAL)

    # value labels
    for bars in (b1, b2):
        for rect in bars:
            h = rect.get_height()
            if np.isfinite(h):
                plt.text(rect.get_x() + rect.get_width()/2, h + max(1, 0.01*h),
                         f"{h:.0f}", ha='center', va='bottom', fontsize=8, color="#444444")

    plt.xticks(x, labels, rotation=30, ha='right')
    plt.ylabel("Throughput (MB/s)")
    plt.title("Side-by-Side Throughput: CephFS vs Local" + subtitle_method)
    plt.legend()
    plt.tight_layout()
    plt.show()

# (Optional) A dumbbell comparison for visual delta on one line per host/label
def dumbbell_local_vs_cephfs(all_df: pd.DataFrame, method: str | None = None):
    """
    Shows, per (hostname[/label]), CephFS vs Local throughput on one line.
    Optional: filter to a single method for a clean read.
    Sorted by speedup (Local/CephFS).
    """
    req_targets = {'cephfs', 'local'}
    if not (set(all_df['target'].unique()) >= req_targets):
        print("[dumbbell] Need both 'cephfs' and 'local'.")
        return

    work = all_df.copy()
    if method is not None:
        work = work[work['method'] == method]
        if work.empty:
            print(f"[dumbbell] No data for method={method!r}.")
            return

    key = ['hostname','label','method']
    ceph = (work[work['target']=='cephfs']
            .groupby(key, as_index=False)['MB_per_s'].mean()
            .rename(columns={'MB_per_s':'cephfs_MBps'}))
    loc  = (work[work['target']=='local']
            .groupby(key, as_index=False)['MB_per_s'].mean()
            .rename(columns={'MB_per_s':'local_MBps'}))

    dfp = pd.merge(ceph, loc, on=key, how='inner')
    if dfp.empty:
        print("[dumbbell] No paired rows after merge.")
        return

    if method is None and dfp['method'].nunique() > 1:
        densest = dfp['method'].value_counts().idxmax()
        dfp = dfp[dfp['method']==densest].copy()
        subtitle_method = f" (method = {densest})"
    else:
        subtitle_method = f" (method = {method})" if method else ""

    dfp['speedup'] = dfp['local_MBps'] / dfp['cephfs_MBps']
    dfp['hostlabel'] = dfp.apply(lambda r: f"{r['hostname']}" + (f" / {r['label']}" if r['label'] else ""), axis=1)
    dfp = dfp.sort_values('speedup', ascending=False)

    labels = dfp['hostlabel'].tolist()
    y = np.arange(len(labels))

    c_ceph = COLOR_CEPH
    c_loc  = COLOR_LOCAL
    c_line = "#bbbbbb"

    plt.figure(figsize=(10, max(4, 0.45 * len(labels))))
    for i, (_, r) in enumerate(dfp.iterrows()):
        x0, x1 = r['cephfs_MBps'], r['local_MBps']
        plt.plot([x0, x1], [i, i], '-', color=c_line, linewidth=2, alpha=0.9)

    plt.scatter(dfp['cephfs_MBps'], y, s=36, label='CephFS (mean MB/s)', color=c_ceph, zorder=3)
    plt.scatter(dfp['local_MBps'],  y, s=36, label='Local (mean MB/s)',  color=c_loc,  zorder=3)

    for i, (_, r) in enumerate(dfp.iterrows()):
        ce, lo = r['cephfs_MBps'], r['local_MBps']
        sp = r['speedup']
        delta = lo - ce
        plt.text(ce, i+0.05, f"{ce:.0f}", va='bottom', ha='right', fontsize=8, color="#444444")
        plt.text(lo, i+0.05, f"{lo:.0f}", va='bottom', ha='left',  fontsize=8, color="#444444")
        xm = (ce + lo) / 2
        plt.text(xm, i-0.12, f"{(sp-1)*100:+.0f}%  Δ={delta:.0f} MB/s", va='top', ha='center', fontsize=8, color="#555555")

    plt.yticks(y, labels)
    plt.xlabel("Throughput (MB/s)")
    plt.title("Local vs CephFS Throughput by Host/Label" + subtitle_method)
    plt.legend(loc='upper right')
    plt.grid(axis='x', linestyle='-', linewidth=0.6, color='#e6e6ee')
    plt.tight_layout()
    plt.show()

# --------------- Plots per target ----------------
for tgt in sorted(df['target'].unique()):
    sub = df[df['target']==tgt]
    grouped_bar_by_host(sub, f"target = {tgt}")
    boxplot_by_method(sub, f"target = {tgt}")

# --------------- Paired comparison (local vs cephfs) ---------------
speed = paired_speedup(df)
if not speed.empty:
    # Show the paired table (Jupyter) and save CSV
    try:
        display(speed.sort_values(['hostname','label','method']))
    except NameError:
        print(speed.sort_values(['hostname','label','method']).to_string(index=False))

    out_csv = RESULTS_DIR / "speedup_summary.csv"
    speed.to_csv(out_csv, index=False)
    print(f"[save] Wrote paired speedup summary: {out_csv}")

    # Clear speedup visualization
    plot_speedup_sorted(speed)

# ---- NEW: Side-by-side Local vs CephFS (choose one method or let it auto-pick densest) ----
# Example 1: auto-pick the method with most paired entries
side_by_side_local_cephfs(df)

# Example 2: focus on a specific method (uncomment one):
# side_by_side_local_cephfs(df, method="dd")
# side_by_side_local_cephfs(df, method="rsync")

# (Optional) Dumbbell comparison (nice visual for deltas)
# dumbbell_local_vs_cephfs(df)              # auto-pick densest method
# dumbbell_local_vs_cephfs(df, method="dd") # or pick a method explicitly

# Quick aggregate table
agg = (df.groupby(['target','method'])['MB_per_s']
         .agg(['count','mean','median','min','max'])
         .reset_index()
         .sort_values(['target','method']))
try:
    display(agg)
except NameError:
    print(agg.to_string(index=False))


## Delete slice

In [ ]:
#slice = fablib.get_slice(name=slice_name)
#slice.delete()